In [1]:
# Εγκατάσταση των απαραίτητων βιβλιοθηκών
import pandas as pd
import numpy as np
import os
import zipfile
from google.colab import files

# Φόρτωση του αρχείου με pandas
df = pd.read_excel("/content/Pre-term-labour-Data-ready-for-ml-pipeline_final.xlsx")

# Εμφάνιση των πρώτων γραμμών του αρχείου για έλεγχο
df.head()

,Maternal age,GA,BW centile,UtA doppler,b-hcg,DVP,MCA doppler,Papp-A,Height,UA doppler,...,Placental location_high posterior with anterior paraplacenta,Placental location_high right,Placental location_ligh anterior with posterior paraplacenta,Placental location_low anterior,Placental location_low posterior,Placental location_low posterior with anterior paraplacenta,Placental location_low right,Placental location_previa,Single umbilical artery_0,Single umbilical artery_1
0,32.000000,24.285714,20.378457,1.010,0.98,5.1,2.03,0.90,166.0,1.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,33.000000,25.285714,16.129898,0.880,0.95,4.0,2.10,1.12,165.0,0.91,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,36.435616,32.571429,74.037300,0.675,1.30,3.6,1.87,0.50,175.0,0.79,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,42.000000,28.714286,71.680853,1.350,1.00,1.4,1.87,0.90,160.0,1.11,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,33.000000,29.285714,22.836726,0.640,1.13,4.1,2.02,1.12,163.0,1.08,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [3]:
# Υπολογισμός του αριθμού των δειγμάτων με τιμή 1 στην στήλη "Preterm_birth"
total_samples = len(df)
samples_with_1 = len(df[df['Preterm_birth'] == 1])

# Όλα τα δείγματα με Preterm_birth == 1
preterm_df = df[df['Preterm_birth'] == 1]

# Υπολογισμός του επιθυμητού αριθμού δειγμάτων με τιμή 1
desired_samples_with_1 = int(0.08 * total_samples)

# Υπολογισμός των δειγμάτων που πρέπει να αφαιρεθούν κάθε φορά
samples_to_remove = samples_with_1 - desired_samples_with_1
print(f"Πρέπει να αφαιρεθούν {samples_to_remove} δείγματα με τιμή '1'.")

Πρέπει να αφαιρεθούν 170 δείγματα με τιμή '1'.


In [10]:
# Φάκελος για τα νέα αρχεία
output_folder = '/content/preterm_dataset_versions'
os.makedirs(output_folder, exist_ok=True)

# Για παρακολούθηση δειγμάτων που έχουν ήδη χρησιμοποιηθεί
used_indices = set()

In [5]:
# Επανάληψη 30 φορές
for i in range(30):
    # Επιλογή από τα μη-χρησιμοποιημένα αν γίνεται
    available = preterm_df[~preterm_df.index.isin(used_indices)]

    # Αν δεν υπάρχουν αρκετά νέα δείγματα, επίτρεψε ξανά δείκτες (για διαφορετικά σύνολα κάθε φορά)
    if len(available) < samples_to_remove:
        available = preterm_df

    # Επιλογή 170 τυχαίων δειγμάτων
    sampled = available.sample(n=samples_to_remove, random_state=np.random.randint(10000))

    # Προσθήκη στα used indices
    used_indices.update(sampled.index)

    # Αφαίρεση από το αρχικό σύνολο
    df_reduced = df.drop(sampled.index)

    # Υπολογισμός νέου ποσοστού
    new_count = len(df_reduced[df_reduced['Preterm_birth'] == 1])
    new_percent = (new_count / len(df_reduced)) * 100

    print(f"[{i+1}/30] Νέο ποσοστό Preterm_birth=1: {new_percent:.2f}%")

    # Αποθήκευση σε νέο Excel αρχείο
    out_path = os.path.join(output_folder, f'reduced_preterm_dataset_{i+1}.xlsx')
    df_reduced.to_excel(out_path, index=False)

[1/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[2/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[3/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[4/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[5/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[6/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[7/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[8/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[9/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[10/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[11/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[12/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[13/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[14/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[15/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[16/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[17/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[18/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[19/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[20/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[21/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[22/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[23/30] Νέο ποσοστό Preterm_birth=1: 8.73%
[24/30] Νέο ποσοστό 

In [11]:
# Δημιουργία zip αρχείου
zip_filename = 'reduced_datasets.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in os.listdir(output_folder):
        zipf.write(os.path.join(output_folder, file))

# Κατέβασμα του zip αρχείου
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>